In [5]:
print("all ok")

all ok


In [25]:
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path, override=True)
    print(f"Loaded .env from: {dotenv_path}")
else:
    print("No .env file found.")

for key in ("OPENAI_API_KEY", "OPENROUTER_API_KEY","HUGGINGFACEHUB_API_TOKEN","GOOGLE_API_KEY"):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    print(f"{key} configured: {bool(value)}")


Loaded .env from: d:\Full_stack_GenAI\Repo\Full-Stack-GenAI\.env
OPENAI_API_KEY configured: True
OPENROUTER_API_KEY configured: True
HUGGINGFACEHUB_API_TOKEN configured: True
GOOGLE_API_KEY configured: True


In [26]:
import os

for key in ("OPENAI_API_KEY", "OPENROUTER_API_KEY","HUGGINGFACEHUB_API_TOKEN","GOOGLE_API_KEY"):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    print(f"{key} configured: {bool(value)}")

OPENAI_API_KEY configured: True
OPENROUTER_API_KEY configured: True
HUGGINGFACEHUB_API_TOKEN configured: True
GOOGLE_API_KEY configured: True


In [23]:
# 👇 UNIVERSAL LLM CALLER
#    One function that works with any of the 6 providers.
#    Fill in the ___ parts using what you learned in Section 1.
from openai import OpenAI
def call_llm(provider: str, prompt: str, api_key: str = "", model: str = "") -> str:
    """
    Call any LLM provider with the same interface.

    Args:
        provider : One of: "ollama" | "lmstudio" | "openai" | "anthropic" | "gemini" | "openrouter"
        prompt   : The question or instruction to send to the model
        api_key  : Your API key (leave empty for local providers Ollama and LM Studio)
        model    : Model name — if empty, a sensible default is used for each provider

    Returns:
        The model's response as a plain Python string
    """
    
    # ------------------------------------------------------------------ #
    #  OPENAI — cloud API at api.openai.com                              #
    # ------------------------------------------------------------------ #
    
    
    if provider == "openai":
        OPEN_API_KEY = os.getenv("OPENAI_API_KEY")
        if not OPEN_API_KEY:
            raise ValueError("OPENAI_API_KEY environment variable not set")
        print("Calling OpenAI...")
        client = OpenAI(api_key=OPEN_API_KEY)   # TODO: pass the api_key parameter to the client
        model = model or "gpt-4o-mini"  # cheapest GPT-4 class model
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500
        )
        return resp.choices[0].message.content
    
    # ------------------------------------------------------------------ #
    #  OPENROUTER — cloud gateway to 200+ models                          #
    # ------------------------------------------------------------------ #
    
    if provider == "openrouter":
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if not OPENROUTER_API_KEY:
            raise ValueError("OPENROUTER_API_KEY environment variable not set")
        print("Calling OpenRouter...")
        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=OPENROUTER_API_KEY,
            default_headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        )
        model = model or "meta-llama/llama-3.3-70b-instruct:free"
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
        )
        return resp.choices[0].message.content
    
    # ------------------------------------------------------------------ #
    #  GEMINI — Google's cloud API, its own SDK                           #
    # ------------------------------------------------------------------ #
    
    elif provider == "gemini":
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
        if not GOOGLE_API_KEY:
            raise ValueError("GOOGLE_API_KEY environment variable not set")
        print("Calling Google Gemini...")
        from google import genai
        client_gemini = genai.Client(api_key=GOOGLE_API_KEY)
        model_name = model or "gemini-2.0-flash"
        response = client_gemini.models.generate_content(
            model=model_name,
            contents=[{"type": "input_text", "input_text": {"text": prompt}}],
            )
        # return the text field (fallback to str(response) if attribute missing)
        return getattr(response, "text", None) or str(response)
    
    else:
        raise ValueError(
            f"Unknown provider: '{provider}'. "
            f"Choose from: ollama, lmstudio, openai, anthropic, gemini, openrouter"
        )


# 👇 Quick smoke test — run this to verify Ollama works
#    (swap "ollama" for "gemini" or "openrouter" if you're on Colab)
result = call_llm("openrouter", "What is 2 + 2? Answer with one word only.")
print(f"Test passed! openai says: {result}")
        
        
        


Calling OpenRouter...


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'meta-llama/llama-3.3-70b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Venice', 'is_byok': False, 'retry_after_seconds': 30, 'retry_after_seconds_raw': 29.362, 'headers': {'Retry-After': '30'}}}, 'user_id': 'user_3DscLzrDSkAtnT5S0djwpy2577Q'}

In [ ]:
import time

# 👇 The SAME question will be sent to every provider you configure below
QUESTION = "What is the most important thing to understand about large language models?"
MY_PROVIDERS = {
    "openai": "gpt-4o-mini",
    "openrouter": "meta-llama/llama-3.3-70b-instruct:free",
    #"gemini": "gemini-2.0-flash"
}

print(f"Question: {QUESTION}")
print("=" * 60)

for provider, model in MY_PROVIDERS.items():
    print(f"\n=== Testing {provider} with model {model} ===")
    start_time = time.time()
    try:
        answer = call_llm(provider, QUESTION, model=model)
        elapsed = time.time() - start_time
        print(f"Answer from {provider} (took {elapsed:.2f} seconds):\n{answer}")
    except Exception as e:
        print(f"Error calling {provider}: {e}")

Question: What is the most important thing to understand about large language models?

=== Testing openrouter with model meta-llama/llama-3.3-70b-instruct:free ===
Calling OpenRouter...
Error calling openrouter: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}


In [16]:
# ─────────────────────────────────────────────────────────────
#  EXPERIMENT: How does TEMPERATURE change the output?
# ─────────────────────────────────────────────────────────────

# Temperature controls how "random" or "creative" the model is:
#   0.0 = very focused, picks the most likely next token every time (deterministic)
#   0.5 = balanced — some creativity, still mostly coherent
#   1.0 = very creative, sometimes surprising, occasionally weird
#   >1.0 = chaotic (usually not useful)

# 👇 But wait — call_llm() doesn't have a temperature parameter yet!
#    This is intentional. For this experiment we call the API directly
#    so we can pass temperature. You'll add it to call_llm() in Section 4.

PROMPT = "Describe artificial intelligence in one sentence. Be creative."

print("Temperature experiment — same prompt, 3 different temperature values:")
print(f"Prompt: '{PROMPT}'\n")

OPEN_API_KEY = os.environ.get("OPEN_API_KEY")

# 👇 We'll use Ollama directly here (change to another provider if needed)
exp_client = OpenAI(api_key=OPEN_API_KEY)   # TODO: pass the api_key parameter to the client
model ="gpt-4o-mini"  # cheapest GPT-4 class model


for temperature in [0.0, 0.5, 1.0]:
    resp = exp_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=temperature,
        max_tokens=500
    )
    answer = resp.choices[0].message.content.strip()
    print(f"Temperature {temperature}: {answer}")
    print()   # blank line between results for readability

# 👇 What to observe:
#    - Temperature 0.0 should give the same (or very similar) answer if you run it twice
#    - Temperature 1.0 should give noticeably different answers on each run
#    - Try running this cell multiple times and compare!

Temperature experiment — same prompt, 3 different temperature values:
Prompt: 'Describe artificial intelligence in one sentence. Be creative.'

Temperature 0.0: Artificial intelligence is the digital sorcery that breathes life into algorithms, transforming mere data into a tapestry of insights, creativity, and decision-making prowess.

Temperature 0.5: Artificial intelligence is like a digital brain, tirelessly weaving patterns of logic and creativity, transforming data into insights and dreams into reality.

Temperature 1.0: Artificial intelligence is the digital sorcery that transforms data into insights, mimicking human thought to illuminate new pathways of innovation and understanding.



In [24]:
# ─────────────────────────────────────────────────────────────
#  EXPERIMENT: How does max_tokens affect the response?
# ─────────────────────────────────────────────────────────────

# max_tokens is a hard cutoff — the model stops generating after this many tokens.
# 1 token ≈ 0.75 words in English.
# Setting it too low can cut off mid-sentence — sometimes comically!

PROMPT_LONG = "Explain how neural networks learn. Be thorough and detailed."

print("max_tokens experiment — same prompt, 3 different length limits:\n")

for max_tok in [20, 100, 500]:
    resp = exp_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": PROMPT_LONG}],
        temperature=0.5,
        max_tokens=max_tok,    # 👈 the variable we're experimenting with
    )
    answer = resp.choices[0].message.content.strip()
    actual_tokens = resp.usage.completion_tokens   # how many tokens were actually generated
    print(f"--- max_tokens={max_tok} (generated {actual_tokens}) ---")
    print(answer)
    print()

max_tokens experiment — same prompt, 3 different length limits:

--- max_tokens=20 (generated 20) ---
Neural networks learn through a process that mimics the way biological brains process information, using a structured

--- max_tokens=100 (generated 100) ---
Neural networks learn through a process that mimics the way the human brain processes information, using a structure of interconnected nodes (neurons) and adjusting the connections (weights) between them based on the data they receive. This learning process primarily involves two key phases: **forward propagation** and **backpropagation**. Let’s dive into each component of this process in detail.

### 1. Structure of Neural Networks

A neural network consists of layers of neurons:

- **Input Layer**:

--- max_tokens=500 (generated 500) ---
Neural networks are a class of models inspired by the biological neural networks that constitute animal brains. They are designed to recognize patterns and make predictions based on input data. 